# Clean Ingestion Notebook — The Penal Code, 1860

This notebook performs only ingestion.

Pipeline:
1. Imports
2. Law Config
3. PDF Load
4. Text Cleaning
5. Section Parsing
6. Chunk Building
7. Sanity Check
8. Chroma Indexing

Notes:
- This notebook does not contain retrieval testing
- This notebook does not contain answer generation
- This notebook does not contain app logic
- It is designed to be law-agnostic with config changes

# 1. Imports

In [80]:
import os
import re
import json
import hashlib
from typing import List, Dict, Any, Optional, Tuple
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

 # 2. Law Config

In [81]:
LAW_ID = "penal_code_1860"
LAW_NAME = "The Penal Code, 1860"
LAW_SHORT_NAME = "Penal Code"

PDF_PATH = "act-print-11.pdf"   # <-- change this if your file name is different
COLLECTION_NAME = "bd_penal_code_1860_en_v1"
PERSIST_DIRECTORY = "chroma_penal_code_1860_v1"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300
MIN_SECTION_TEXT_LENGTH = 40
MAX_REASONABLE_SECTION = 600

# 3. PDF Load

In [82]:
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def load_pdf_pages(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    return loader.load()


documents = load_pdf_pages(PDF_PATH)

provenance = {
    "file_path": PDF_PATH,
    "file_name": os.path.basename(PDF_PATH),
    "sha256": sha256_file(PDF_PATH),
    "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_url": None,
    "act_name": LAW_NAME,
}

print("Loaded pages:", len(documents))
print("PDF exists:", os.path.exists(PDF_PATH))
print("First page preview:\n")
print(documents[0].page_content[:1000])

Loaded pages: 200
PDF exists: True
First page preview:

04/03/2026 The Penal Code, 1860
CHAPTER I
INTRODUCTION
The Penal Code, 1860
( ACT NO. XLV OF 1860 )
1
[ 6th October, 1860 ]
Preamble WHEREAS it is expedient to provide a general Penal Code for Bangladesh;
It is enacted as follows:-
Title and
extent of
operation of
the Code
1. This Act shall be called the [Penal Code], and shall take effect throughout
Bangladesh.
2
Punishment
of offences
committed
within
Bangladesh
2. Every person shall be liable to punishment under this Code and not
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
Punishment
of offences
committed
beyond, but
which by
law may be
tried within
Bangladesh
3. Any person liable, by any Bangladesh Law, to be tried for an offence
committed beyond Bangladesh shall be dealt with according to the provisions
of this Code for any act committed beyond Bangladesh in the same manner
as if such act had been comm

# 4. Text Cleaning

In [83]:
def normalize_legal_text(page_text: str) -> str:
    if not page_text:
        return ""

    text = page_text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove known footer/site noise
    text = re.sub(r"(?i)\bbdlaws\.minlaw\.gov\.bd/act-print-\d+\.html\b", "", text)
    text = re.sub(r"(?m)^\s*\d{1,4}/\d{1,4}\s*$", "", text)

# Fix glued section starts like "Short title1." or "etc.380." -> newline before section number
    text = re.sub(r"([A-Za-z\)\-.,;])(\d{1,4}\.)", r"\1\n\2 ", text)
    text = re.sub(r"([A-Za-z\]\)])\s+(\d{1,4}\.)", r"\1\n\2 ", text)

# Fix glued heading/body like "CommencementIt extends..." -> "Commencement\nIt extends..."
    text = re.sub(r"([a-z\)])([A-Z])", r"\1\n\2", text)
    text = re.sub(r"(?m)^([A-Za-z][A-Za-z\-\s]+)\s+(\d{1,4}\.)", r"\1\n\2 ", text)

# Normalize spaces/newlines
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()


def build_clean_pages(documents) -> List[Dict[str, Any]]:
    clean_pages = []
    for i, doc in enumerate(documents):
        clean_pages.append({
            "page": i,
            "text": normalize_legal_text(doc.page_content),
        })
    return clean_pages


clean_pages = build_clean_pages(documents)

print("Cleaned page 0 preview:\n")
print(clean_pages[0]["text"][:2500])

Cleaned page 0 preview:

04/03/2026 The Penal Code, 1860
CHAPTER I
INTRODUCTION
The Penal Code, 1860
( ACT NO. XLV OF 1860 )
1
[ 6th October, 1860 ]
Preamble WHEREAS it is expedient to provide a general Penal Code for Bangladesh;
It is enacted as follows:-
Title and
extent of
operation of
the Code
1. This Act shall be called the [Penal Code], and shall take effect throughout
Bangladesh.
2
Punishment
of offences
committed
within
Bangladesh
2. Every person shall be liable to punishment under this Code and not
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
Punishment
of offences
committed
beyond, but
which by
law may be
tried within
Bangladesh
3. Any person liable, by any Bangladesh Law, to be tried for an offence
committed beyond Bangladesh shall be dealt with according to the provisions
of this Code for any act committed beyond Bangladesh in the same manner
as if such act had been committed within Bangladesh.
Extens

In [84]:
def inspect_page(page_num: int, chars: int = 2500):
    print("=" * 120)
    print(f"PAGE: {page_num}")
    print("=" * 120)
    print(clean_pages[page_num]["text"][:chars])


for p in [0, 1, 2, 3, 4, 5, 10, 20, 40, 60]:
    inspect_page(p, chars=2500)

PAGE: 0
04/03/2026 The Penal Code, 1860
CHAPTER I
INTRODUCTION
The Penal Code, 1860
( ACT NO. XLV OF 1860 )
1
[ 6th October, 1860 ]
Preamble WHEREAS it is expedient to provide a general Penal Code for Bangladesh;
It is enacted as follows:-
Title and
extent of
operation of
the Code
1. This Act shall be called the [Penal Code], and shall take effect throughout
Bangladesh.
2
Punishment
of offences
committed
within
Bangladesh
2. Every person shall be liable to punishment under this Code and not
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
Punishment
of offences
committed
beyond, but
which by
law may be
tried within
Bangladesh
3. Any person liable, by any Bangladesh Law, to be tried for an offence
committed beyond Bangladesh shall be dealt with according to the provisions
of this Code for any act committed beyond Bangladesh in the same manner
as if such act had been committed within Bangladesh.
Extension of
Code to
ex

# 5. Section Parsing

In [85]:
PAGE_MARKER_TEMPLATE = "\n\n<<<PAGE:{page}>>>\n\n"


def build_full_text(clean_pages: List[Dict[str, Any]]) -> str:
    parts = []
    for item in clean_pages:
        parts.append(PAGE_MARKER_TEMPLATE.format(page=item["page"]) + item["text"])
    return "\n".join(parts)


def build_page_ranges(full_text: str) -> List[Tuple[int, int, int]]:
    matches = list(re.finditer(r"<<<PAGE:(\d+)>>>", full_text))
    ranges = []

    for idx, m in enumerate(matches):
        page_num = int(m.group(1))
        start = m.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(full_text)
        ranges.append((page_num, start, end))

    return ranges


def offset_to_page(offset: int, page_ranges: List[Tuple[int, int, int]]) -> Optional[int]:
    for page_num, start, end in page_ranges:
        if start <= offset < end:
            return page_num
    return None


def parse_sections(full_text: str) -> Dict[int, Dict[str, Any]]:
    lines = full_text.splitlines()
    sections = {}

    current_section = None
    current_heading = None
    current_lines = []
    current_start_offset = None

    pending_heading_lines = []
    offset = 0

    section_start_re = re.compile(r"^\s*(\d{1,4})\.\s*(.+?)\s*$")

    def is_skippable_line(stripped: str) -> bool:
        if not stripped:
            return False
        if stripped.startswith("<<<PAGE:"):
            return False
        if "bdlaws.minlaw.gov.bd" in stripped.lower():
            return True
        if re.match(r"^\d{1,4}/\d{1,4}$", stripped):
            return True
        if re.match(r"^\d{1,4}$", stripped):
            return True
        if stripped == LAW_NAME or stripped.endswith(LAW_NAME):
            return True
        if stripped.startswith("(") and "ACT NO." in stripped.upper():
            return True
        if re.match(r"^\[\s*\d{1,2}[a-z]{2}.+\]$", stripped, re.IGNORECASE):
            return True
        if stripped.lower() in {"preamble", "preliminary"}:
            return True
        return False

    def looks_like_heading_fragment(stripped: str) -> bool:
        if not stripped:
            return False
        if re.match(r"^\d{1,4}$", stripped):
            return False
        if len(stripped) > 60:
            return False
        if "," in stripped:
            return False
        if re.search(r"[.;:]$", stripped):
            return False
        if re.search(r"\d", stripped):
            return False

        words = stripped.split()
        if len(words) > 6:
            return False

        allowed_small = {"of", "the", "and", "or", "for", "in", "to", "by", "with", "etc"}
        alpha_words = [w.strip("[](),.-") for w in words if re.search(r"[A-Za-z]", w)]
        if not alpha_words:
            return False

        return all(
            (w and w[0].isupper()) or (w.lower() in allowed_small)
            for w in alpha_words
        )

    for line in lines:
        raw_line = line
        stripped = raw_line.strip()

        line_start_offset = offset
        offset += len(raw_line) + 1

        if not stripped:
            if current_section is not None:
                current_lines.append(raw_line)
            continue

        if stripped.startswith("<<<PAGE:"):
            if current_section is not None:
                current_lines.append(raw_line)
            continue

        if is_skippable_line(stripped):
            continue

        m = section_start_re.match(stripped)

        if m:
            sec_num = int(m.group(1))
            inline_heading = m.group(2).strip()

            if 0 < sec_num <= MAX_REASONABLE_SECTION and (
                current_section is None or sec_num > current_section
            ):
                combined_heading_parts = pending_heading_lines + [inline_heading]
                combined_heading = " ".join(combined_heading_parts).strip()
                combined_heading = re.sub(r"\s+", " ", combined_heading)

                if current_section is not None and pending_heading_lines:
                    while current_lines and any(
                        current_lines[-1].strip() == h for h in pending_heading_lines
                ):
                        current_lines.pop()

                if current_section is not None:
                    current_text = "\n".join(current_lines).strip()
                    sections[current_section] = {
                        "section_number": current_section,
                        "heading": current_heading,
                        "text": current_text,
                        "start_offset": current_start_offset,
                        "end_offset": line_start_offset,
                    }

                current_section = sec_num
                current_heading = combined_heading
                current_lines = []
                current_start_offset = line_start_offset
                pending_heading_lines = []
                continue

        if looks_like_heading_fragment(stripped):
            pending_heading_lines.append(stripped)
            continue

        pending_heading_lines = []

        if current_section is not None:
            current_lines.append(raw_line)

    if current_section is not None:
        sections[current_section] = {
            "section_number": current_section,
            "heading": current_heading,
            "text": "\n".join(current_lines).strip(),
            "start_offset": current_start_offset,
            "end_offset": len(full_text),
        }

    return sections


full_text = build_full_text(clean_pages)
page_ranges = build_page_ranges(full_text)
sections = parse_sections(full_text)

for sec, obj in sections.items():
    obj["page_start"] = offset_to_page(obj["start_offset"], page_ranges)
    obj["page_end"] = offset_to_page(max(obj["start_offset"], obj["end_offset"] - 1), page_ranges)

print("Parsed sections:", len(sections))
print("First 20 section numbers:", list(sorted(sections.keys()))[:20])

Parsed sections: 489
First 20 section numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 15, 16, 18, 20, 22, 23, 27, 30]


In [86]:
def inspect_section_raw(sections: Dict[int, Dict[str, Any]], sec_num: int, preview_chars: int = 2000):
    if sec_num not in sections:
        print(f"Section {sec_num} NOT FOUND")
        return

    obj = sections[sec_num]
    print("=" * 120)
    print("SECTION:", obj["section_number"])
    print("HEADING:", obj["heading"])
    print("PAGES:", obj["page_start"], "-", obj["page_end"])
    print("-" * 120)
    print(obj["text"][:preview_chars])


inspect_section_raw(sections, 1, preview_chars=1500)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 2, preview_chars=2000)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 3, preview_chars=1200)

SECTION: 1
HEADING: the Code This Act shall be called the [Penal Code], and shall take effect throughout
PAGES: 0 - 0
------------------------------------------------------------------------------------------------------------------------
Bangladesh.
of offences
committed
within

########################################################################################################################

SECTION: 2
HEADING: Bangladesh Every person shall be liable to punishment under this Code and not
PAGES: 0 - 0
------------------------------------------------------------------------------------------------------------------------
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
of offences
committed
beyond, but
which by
law may be
tried within

########################################################################################################################

SECTION: 3
HEADING: Bangladesh Any person liable, by any

In [87]:
def find_raw_section_window(clean_pages, target: str, page_start: int = 130, page_end: int = 142, window: int = 1200):
    found = False

    for i in range(page_start, page_end + 1):
        text = clean_pages[i]["text"]
        idx = text.find(target)
        if idx != -1:
            found = True
            print("=" * 120)
            print("PAGE:", i)
            print("TARGET:", target)
            print("-" * 120)
            start = max(0, idx - 300)
            end = min(len(text), idx + window)
            print(text[start:end])

    if not found:
        print(f"{target!r} not found in pages {page_start}-{page_end}")


find_raw_section_window(clean_pages, "379.", 130, 142)
print("\n" + "#" * 120 + "\n")
find_raw_section_window(clean_pages, "380.", 130, 142)

PAGE: 139
TARGET: 379.
------------------------------------------------------------------------------------------------------------------------
he has not
authority from Z to give. If A takes the property dishonestly, he commits theft.
(p) A, in good faith, believing property belonging to Z to be A's own property,
takes that property out of B's possession. Here, as A does not take
dishonestly, he does not commit theft.
Punishment
for theft
379. Whoever commits theft shall be punished with imprisonment of either
description for a term which may extend to three years, or with fine, or with
both.
Theft in
dwelling-
house, etc.
380. Whoever commits theft in any building, tent or vessel, which building,
tent or vessel is used as a human dwelling, or use for the custody of
property, shall be punished with imprisonment of either description for a term
which may extend to seven years, and shall also be liable to fine.
Theft by
clerk or

#########################################################

# 6. Chunk Building

In [88]:
PAGE_TAG_RE = re.compile(r"<<<PAGE:(\d+)>>>")

def strip_page_tags(text: str) -> str:
    text = PAGE_TAG_RE.sub("", text)

    # Remove obvious running headers / dates
    text = re.sub(r"(?m)^\d{2}/\d{2}/\d{4}\s+The Contract Act,\s*1872\s*$", "", text)

    # Remove short marginal-heading lines (layout noise)
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            cleaned_lines.append("")
            continue

        # Drop short title-like labels (Extent, Commencement, etc.)
        if len(stripped) <= 35 and re.match(r"^[A-Za-z][A-Za-z\-\s]*$", stripped):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def build_documents(sections: Dict[int, Dict[str, Any]], provenance: Dict[str, Any]) -> List[Document]:
    docs_for_db = []
    doc_sha = provenance["sha256"]

    for sec_num in sorted(sections.keys()):
        sec_obj = sections[sec_num]
        clean_text = strip_page_tags(sec_obj["text"])

        if len(clean_text) < MIN_SECTION_TEXT_LENGTH:
            continue

        chunks = splitter.split_text(clean_text)

        for idx, chunk_text in enumerate(chunks):
            metadata = {
                "law_id": LAW_ID,
                "law_name": LAW_NAME,
                "law_short_name": LAW_SHORT_NAME,
                "act_name": LAW_NAME,
                "section_number": sec_num,
                "section_heading": sec_obj["heading"],
                "page_start": sec_obj["page_start"],
                "page_end": sec_obj["page_end"],
                "source_pdf": provenance["file_name"],
                "source_sha256": doc_sha,
                "chunk_index": idx,
                "chunk_id": f"{doc_sha[:12]}::{LAW_ID}::{sec_num}::{idx}",
            }

            docs_for_db.append(Document(page_content=chunk_text, metadata=metadata))

    return docs_for_db


docs_for_db = build_documents(sections, provenance)

print("Total chunks prepared:", len(docs_for_db))
print("Sample metadata:\n", docs_for_db[0].metadata if docs_for_db else "No docs")
print("\nSample text:\n")
print(docs_for_db[0].page_content[:500] if docs_for_db else "No docs")

Total chunks prepared: 529
Sample metadata:
 {'law_id': 'penal_code_1860', 'law_name': 'The Penal Code, 1860', 'law_short_name': 'Penal Code', 'act_name': 'The Penal Code, 1860', 'section_number': 2, 'section_heading': 'Bangladesh Every person shall be liable to punishment under this Code and not', 'page_start': 0, 'page_end': 0, 'source_pdf': 'act-print-11.pdf', 'source_sha256': '270272b919253793b062975a8b92fac2a62aa3a78fba94ea374202ec52acb5d3', 'chunk_index': 0, 'chunk_id': '270272b91925::penal_code_1860::2::0'}

Sample text:

otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
beyond, but


# 7. Sanity Check

In [89]:
def preview_chunks(docs: List[Document], n: int = 3):
    for i, doc in enumerate(docs[:n], start=1):
        md = doc.metadata
        print("=" * 100)
        print(f"Chunk #{i}")
        print("Section:", md.get("section_number"))
        print("Heading:", md.get("section_heading"))
        print("Pages:", md.get("page_start"), "-", md.get("page_end"))
        print("Text preview:")
        print(doc.page_content[:800])
        print()


preview_chunks(docs_for_db, n=3)

Chunk #1
Section: 2
Heading: Bangladesh Every person shall be liable to punishment under this Code and not
Pages: 0 - 0
Text preview:
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
beyond, but

Chunk #2
Section: 3
Heading: Bangladesh Any person liable, by any Bangladesh Law, to be tried for an offence
Pages: 0 - 0
Text preview:
committed beyond Bangladesh shall be dealt with according to the provisions
of this Code for any act committed beyond Bangladesh in the same manner
as if such act had been committed within Bangladesh.

Chunk #3
Section: 4
Heading: The provisions of this Code apply also to any offence committed by-
Pages: 0 - 1
Text preview:
(1) any citizen of Bangladesh in any place without and beyond Bangladesh;

(2) [Omitted by Article 2 and Schedule of the Central Laws (Adaptation)
Order, 1961.]
(3) [Omitted by section 3 and 2nd Schedule of the Bangladesh Laws
(4) any person on any ship or aircraft regist

In [90]:
def inspect_one_section(sections: Dict[int, Dict[str, Any]], sec: int, preview_chars: int = 1200):
    print("\n" + "=" * 120)
    if sec not in sections:
        print(f"Section {sec} NOT FOUND")
        return

    obj = sections[sec]
    print("SECTION:", obj["section_number"])
    print("HEADING:", obj["heading"])
    print("PAGES:", obj["page_start"], "-", obj["page_end"])
    print("-" * 120)
    print(obj["text"][:preview_chars])


inspect_one_section(sections, 379, preview_chars=1000)
inspect_one_section(sections, 380, preview_chars=1000)


SECTION: 379
HEADING: Whoever commits theft shall be punished with imprisonment of either
PAGES: 139 - 139
------------------------------------------------------------------------------------------------------------------------
description for a term which may extend to three years, or with fine, or with
both.
dwelling-
house, etc.

SECTION: 380
HEADING: Whoever commits theft in any building, tent or vessel, which building,
PAGES: 139 - 140
------------------------------------------------------------------------------------------------------------------------
tent or vessel is used as a human dwelling, or use for the custody of
property, shall be punished with imprisonment of either description for a term
which may extend to seven years, and shall also be liable to fine.
clerk or


<<<PAGE:140>>>

servant of
property in
possession
of master


In [91]:
def inspect_key_sections(sections: Dict[int, Dict[str, Any]], keys, preview_chars: int = 1200):
    for sec in keys:
        print("\n" + "=" * 120)
        if sec not in sections:
            print(f"Section {sec} NOT FOUND")
            continue

        obj = sections[sec]
        print("SECTION:", obj["section_number"])
        print("HEADING:", obj["heading"])
        print("PAGES:", obj["page_start"], "-", obj["page_end"])
        print("-" * 120)
        print(obj["text"][:preview_chars])


inspect_key_sections(sections, [378, 379, 380])


SECTION: 378
HEADING: Theft Whoever, intending to take dishonestly any moveable property out of
PAGES: 136 - 139
------------------------------------------------------------------------------------------------------------------------
the possession of any person without that person's consent, moves that
property in order to such taking, is said to commit theft.


<<<PAGE:137>>>

04/03/2026 The Penal Code, 1860 Explanation
1. -A thing so long as it is attached to the earth, not being
moveable property, is not the subject of theft; but it becomes capable of
being the subject of theft as soon as it is severed from the earth.
2. -A moving effected by the same act which effects the
severance may be a theft.
3. -A person is said to cause a thing to move by removing an
obstacle which prevented it from moving or by separating it from any other
thing, as well as by actually moving it.
4. -A person, who by any means causes an animal to move, is
said to move that animal, and to move everything w

In [92]:
def inspect_sections(sections: Dict[int, Dict[str, Any]], section_numbers=None, preview_chars: int = 700):
    if section_numbers is None:
        section_numbers = list(sorted(sections.keys()))[:10]

    for sec in section_numbers:
        if sec not in sections:
            print(f"\nSection {sec} NOT FOUND")
            continue

        obj = sections[sec]
        print("\n" + "=" * 110)
        print(f"Section: {obj.get('section_number')}")
        print(f"Heading: {obj.get('heading')}")
        print(f"Pages: {obj.get('page_start')} - {obj.get('page_end')}")
        print("Preview:")
        print(obj.get("text", "")[:preview_chars])


inspect_sections(
    sections,
    section_numbers=[1, 2, 3, 4, 10]
)

print("\n" + "#" * 110 + "\n")

preview_chunks(docs_for_db, n=5)


Section: 1
Heading: the Code This Act shall be called the [Penal Code], and shall take effect throughout
Pages: 0 - 0
Preview:
Bangladesh.
of offences
committed
within

Section: 2
Heading: Bangladesh Every person shall be liable to punishment under this Code and not
Pages: 0 - 0
Preview:
otherwise for every act or omission contrary to the provisions thereof, of
which he shall be guilty within Bangladesh.
of offences
committed
beyond, but
which by
law may be
tried within

Section: 3
Heading: Bangladesh Any person liable, by any Bangladesh Law, to be tried for an offence
Pages: 0 - 0
Preview:
committed beyond Bangladesh shall be dealt with according to the provisions
of this Code for any act committed beyond Bangladesh in the same manner
as if such act had been committed within Bangladesh.
extra-

Section: 4
Heading: The provisions of this Code apply also to any offence committed by-
Pages: 0 - 1
Preview:
(1) any citizen of Bangladesh in any place without and beyond Bangladesh;


<<<PAG

# 8. Chroma Indexing

In [93]:
embedding = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectordb = Chroma.from_documents(
    documents=docs_for_db,
    embedding=embedding,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIRECTORY,
)

vectordb.persist()

print("Chroma persisted successfully.")
print("Collection:", COLLECTION_NAME)
print("Persist directory:", PERSIST_DIRECTORY)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma persisted successfully.
Collection: bd_penal_code_1860_en_v1
Persist directory: chroma_penal_code_1860_v1


# checking

In [ ]:
print("=" * 100)
print("INGESTION SUMMARY")
print("=" * 100)
print("LAW_ID:", LAW_ID)
print("LAW_NAME:", LAW_NAME)
print("PDF_PATH:", PDF_PATH)
print("Loaded pages:", len(documents))
print("Parsed sections:", len(sections))
print("Prepared chunks:", len(docs_for_db))
print("Collection:", COLLECTION_NAME)
print("Persist directory:", PERSIST_DIRECTORY)

if docs_for_db:
    print("\nSample final chunk metadata:")
    print(docs_for_db[0].metadata)

print("\nNotebook ingestion pipeline completed successfully.")

INGESTION SUMMARY
LAW_ID: penal_code_1860
LAW_NAME: The Penal Code, 1860
PDF_PATH: act-print-11.pdf
Loaded pages: 200
Parsed sections: 489
Prepared chunks: 520
Collection: bd_penal_code_1860_en_v1
Persist directory: chroma_penal_code_1860_v1

Sample final chunk metadata:
{'law_id': 'penal_code_1860', 'law_name': 'The Penal Code, 1860', 'law_short_name': 'Penal Code', 'act_name': 'The Penal Code, 1860', 'section_number': 1, 'section_heading': '-It may amount to defamation to impute any thing to a', 'page_start': 182, 'page_end': 182, 'source_pdf': 'act-print-11.pdf', 'source_sha256': '270272b919253793b062975a8b92fac2a62aa3a78fba94ea374202ec52acb5d3', 'chunk_index': 0, 'chunk_id': '270272b91925::penal_code_1860::1::0'}

Notebook ingestion pipeline completed successfully.


: 

: 

In [ ]:
inspect_section_raw(sections, 378, preview_chars=1500)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 379, preview_chars=1500)

SECTION: 378
HEADING: Theft Whoever, intending to take dishonestly any moveable property out of
PAGES: 136 - 139
------------------------------------------------------------------------------------------------------------------------
property in order to such taking, is said to commit theft.


<<<PAGE:137>>>

04/03/2026 The Penal Code, 1860 Explanation
1. -A thing so long as it is attached to the earth, not being
being the subject of theft as soon as it is severed from the earth.
2. -A moving effected by the same act which effects the
severance may be a theft.
3. -A person is said to cause a thing to move by removing an
thing, as well as by actually moving it.
4. -A person, who by any means causes an animal to move, is
the motion so caused, is moved by that animal.
5. -The consent mentioned in the definition may be express or
person having for that purpose authority either express or implied.
severed the tree in order to such taking, he has committed theft.
(b) A puts a bait for dogs i

: 

In [ ]:
inspect_section_raw(sections, 1, preview_chars=1200)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 2, preview_chars=1200)
print("\n" + "#" * 120 + "\n")
inspect_section_raw(sections, 300, preview_chars=1200)

SECTION: 1
HEADING: Explanation -It may amount to defamation to impute any thing to a
PAGES: 182 - 182
------------------------------------------------------------------------------------------------------------------------
if living, and is intended to be hurtful to the feelings of his family or other near
relatives.

########################################################################################################################

SECTION: 2
HEADING: Explanation -It may amount to defamation to make an imputation
PAGES: 182 - 182
------------------------------------------------------------------------------------------------------------------------
concerning a company or an association or collection of persons as such.

########################################################################################################################

SECTION: 300
HEADING: Murder Except in the cases hereinafter excepted, culpable homicide is murder,
PAGES: 108 - 110
-----------------------

: 

In [ ]:
obj = sections[379]
print("HEADING:", obj["heading"])
print("TEXT LENGTH:", len(obj["text"]))
print(obj["text"][:2000])

HEADING: Punishment for theft Whoever commits theft shall be punished with imprisonment of either
TEXT LENGTH: 17
both.
house, etc.


: 

In [ ]:
obj = sections[379]
print("HEADING:", obj["heading"])
print("TEXT LENGTH:", len(obj["text"]))
print(obj["text"][:500])

HEADING: Whoever commits theft shall be punished with imprisonment of either
TEXT LENGTH: 105
description for a term which may extend to three years, or with fine, or with
both.
dwelling-
house, etc.


: 

In [ ]:
inspect_page(136, chars=2500)
print("\n" + "#" * 120 + "\n")
inspect_page(137, chars=2500)
print("\n" + "#" * 120 + "\n")
inspect_page(138, chars=2500)

PAGE: 136
04/03/2026 The Penal Code, 1860
Of Unnatural Offences
CHAPTER XVII
OF OFFENCES AGAINST PROPERTY
Of Theft
Fourthly. With her consent, when the man knows that he is not her husband,
and that her consent is given because she believes that he is another man to
whom she is or believes herself to be lawfully married.
Fifthly. With or without her consent, when she is under fourteen years of age.
Explanation. Penetration is sufficient to constitute the sexual intercourse
necessary to the offence of rape.
Exception. Sexual intercourse by a man with his own wife, the wife not being
under thirteen years of age, is not rape.
Punishment
for rape
376. Whoever commits rape shall be punished with [imprisonment] for life
or with imprisonment of either description for a term which may extend to ten
years, and shall also be liable to fine, unless the woman raped is his own
wife and is not under twelve years of age, in which case he shall be punished
with imprisonment of either description for a

: 

: 

In [ ]:
inspect_section_raw(sections, 376)
inspect_section_raw(sections, 377)
inspect_section_raw(sections, 378)
inspect_section_raw(sections, 379)

SECTION: 376
HEADING: Punishment for rape Whoever commits rape shall be punished with [imprisonment] for life
PAGES: 136 - 136
------------------------------------------------------------------------------------------------------------------------
years, or with fine, or with both.
SECTION: 377
HEADING: Unnatural offences Whoever voluntarily has carnal intercourse against the order of nature
PAGES: 136 - 136
------------------------------------------------------------------------------------------------------------------------
described in this section.
SECTION: 378
HEADING: Theft Whoever, intending to take dishonestly any moveable property out of
PAGES: 136 - 137
------------------------------------------------------------------------------------------------------------------------
property in order to such taking, is said to commit theft.


<<<PAGE:137>>>

04/03/2026 The Penal Code, 1860 Explanation
SECTION: 379
HEADING: Punishment for theft Whoever commits theft shall be punished wi

: 

: 

In [ ]:
inspect_section_raw(sections, 379, preview_chars=2000)
print("\n" + "#" * 120 + "\n")
inspect_page(139, chars=2500)

SECTION: 379
HEADING: Punishment for theft Whoever commits theft shall be punished with imprisonment of either
PAGES: 139 - 139
------------------------------------------------------------------------------------------------------------------------
both.
house, etc.

########################################################################################################################

PAGE: 139
04/03/2026 The Penal Code, 1860
commits theft, though the watch is his own property inasmuch as he takes it
dishonestly.
(l) A takes an article belonging to Z out of Z's possession without Z's consent,
with the intention of keeping it until he obtains money from Z as a reward for
its restoration. Here A takes dishonestly; A has therefore committed theft.
(m) A, being on friendly terms with Z, goes into Z's library in Z's absence, and
takes away a book without Z's express consent for the purpose merely of
reading it, and with the intention of returning it. Here, it is probable that A may
have c

: 

: 

In [ ]:
page_136 = clean_pages[136]["text"]
idx = page_136.find("378.")
print(page_136[max(0, idx-500): idx+1200])

a term which may extend to two
years, or with fine, or with both.
122
Unnatural
offences
377. Whoever voluntarily has carnal intercourse against the order of nature
with any man, woman or animal, shall be punished with [imprisonment] for
life, or with imprisonment of either description for a term which may extend to
ten years, and shall also be liable to fine.Explanation. Penetration is
sufficient to constitute the carnal intercourse necessary to the offence
described in this section.
123
Theft
378. Whoever, intending to take dishonestly any moveable property out of
the possession of any person without that person's consent, moves that
property in order to such taking, is said to commit theft.


: 

: 

In [ ]:
page_139 = clean_pages[139]["text"]
idx = page_139.find("379.")
print(page_139[max(0, idx-300): idx+1200])

he has not
authority from Z to give. If A takes the property dishonestly, he commits theft.
(p) A, in good faith, believing property belonging to Z to be A's own property,
takes that property out of B's possession. Here, as A does not take
dishonestly, he does not commit theft.
Punishment
for theft
379. Whoever commits theft shall be punished with imprisonment of either
description for a term which may extend to three years, or with fine, or with
both.
Theft in
dwelling-
house, etc.
380. Whoever commits theft in any building, tent or vessel, which building,
tent or vessel is used as a human dwelling, or use for the custody of
property, shall be punished with imprisonment of either description for a term
which may extend to seven years, and shall also be liable to fine.
Theft by
clerk or


: 

: 